# [LAB-03] 4. 모델링 - 연습문제

## 📚 모델을 고를 것인가, 다듬을 것인가

### 당신은 지난 단원에서 투입 변수 9종을 확정한 그 분석가입니다.

로그 변환과 라벨링까지 끝낸 최종 데이터셋 21,600건을 들고 갔더니 팀장이 말합니다. "변수 얘기는 이제 됐다. 그럼 실제로 값을 맞혀 봐라. 선형부터 앙상블까지 **한 번에 다 돌려서** 어느 쪽이 이 데이터에 맞는지 보고해라. 모델 하나 붙잡고 우기지 말고."

이어서 조건을 답니다. "**8 대 2**로 나눠서 학습에 쓰지 않은 데이터로만 평가해라. 주 지표는 **RMSE**, 보조로 **MAE와 결정계수**를 같이 봐라. 그리고 1등 하나만 적어 오지 마라. 소수점 셋째 자리에서 갈리는 걸 1등이라고 우기는 건 의미가 없으니, **1등과 사실상 차이가 없는 후보들까지 묶어서** 가져와라."

마지막 요구가 남았습니다. "기본값으로 한 번 돌린 게 끝이 아니다. **하이퍼파라미터를 격자로 훑으면서 5겹 교차검증**을 걸어 다시 돌려라. 튜닝하면 다 좋아지는지, **오히려 나빠지는 모델은 없는지** 그것까지 확인해서 최종적으로 무엇을 쓸지 답을 가져와라."

투입 모델은 선형 4종 · 비선형 2종 · 트리 1종 · 앙상블 4종, 모두 **11개**입니다. 종속변수는 로그 변환된 주택 가격이고, 독립변수 9종에는 범주형 3종(리모델링 여부 · 지하 여부 · 공간 클러스터)이 섞여 있습니다. 그래서 다중공선성 제거 · 정규화 · 더미 인코딩은 모델 성격에 맞춰 **파이프라인 안에** 넣어 학습과 함께 돌립니다. 분리와 학습의 난수 씨앗은 모두 고정합니다.

## #01. 준비작업

### 1. 패키지 설치

> 이미 모두 설치했으므로 생략

### 2. 기본 라이브러리 참조

In [6]:
from jussam import load_data
from helpers import *
import datetime as dt
import os
import glob as gl   # 파일 목록을 리스트로 반환하는 파이썬 내장 모듈

# 훈련,검증 데이터 분리 함수
from sklearn.model_selection import train_test_split

# 하이퍼파라미터 튜닝
from sklearn.model_selection import GridSearchCV

### 3. 머신러닝 학습 모델 라이브러리 참조

In [7]:
# 선형 계열 모델
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# 비선형 계열 모델
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# 트리계열 모델
from sklearn.tree import DecisionTreeRegressor

# 앙상블 모델
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

### 4. 데이터 불러오기

- 파생변수를 포함한 최종 변수 채택 후 로그 변환과 라벨링이 완료된 데이터

In [8]:
# 분석용 데이터 전처리가 완료된 데이터 셋
origin = load_data("kc_house_features_log_labelled")
df = my_qtcheck.set_type(origin, as_category=["is_renovated", "has_basement", "spatial_cluster"])

📚 캘리포니아 주택 가격 데이터셋의 파생변수 추가 버전에 로그 변환 및 라벨링 적용 (출처: 자체 작업)

    field               description
--  ------------------  -------------------------------------
 0  grade               건축 등급 점수 (1~13, 로그변환됨)
 1  view                조망 점수 (0~4, log1p 변환됨)
 2  sqft_per_bedroom    침실 하나당 거주 면적 (로그변환됨)
 3  living_ratio        대지 대비 거주 면적 (로그변환됨)
 4  above_ratio         거주 면적 중 지상 비율
 5  living_vs_neighbor  인근 15채 대비 거주 면적 (로그변환됨)
 6  is_renovated        리모델링 여부 (이진형: 1/0)
 7  has_basement        지하 여부 (이진형: 1/0)
 8  spatial_cluster     공간 클러스터
 9  price               주택 가격 (USD 달러, 로그변환됨)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21600 entries, 0 to 21599
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   grade               21600 non-null  float64 
 1   view                21600 non-null  float64 
 2   sqft_per_bedroom    21600 non-null  float64 
 3   living_ratio        21600 non-null  float64 
 

### 5. 데이터에서 독립변수와 종속변수 분리

- 머신러닝에서는 독립변수를 feature, 종속변수를 target으로 명명한다.

In [9]:
feature = df.drop(columns=["price"])
target = df["price"]
feature.shape, target.shape

((21600, 9), (21600,))

### 6. 훈련, 검증 데이터 분리

In [10]:
# 훈련, 검증 데이터 분리
# 분류 문제인 경우 파라미터 추가 --> stratify=target
x_train, x_test, y_train, y_test = train_test_split(feature, target, test_size=0.2, random_state=RANDOM_STATE)

x_train.shape, x_test.shape, y_train.shape, y_test.shape

((17280, 9), (4320, 9), (17280,), (4320,))

### 7. 학습을 완료한 모델이 저장될 폴더

In [11]:
timestemp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
workdir = f"kc_house/{timestemp}"

if not os.path.exists(workdir):
    os.makedirs(workdir)

## #02. 선형계열 모델

### 1. LinearRegressor

In [12]:
# 파이프라인 구축 -> 학습 -> 저장
linear = my_ml.fit_pipeline(
                model=LinearRegression(),  # <-- LinearRegressor 학습 모델
                x_train=x_train, y_train=y_train, 
                vif=True, 
                scale=False, 
                drop_first=True,
                save_path=f"{workdir}/linear.pkl")

linear

대상: 17280행 x 9열 | 모델: LinearRegression
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0)
  명목형: 더미변수 인코딩(drop_first=True)

모델 학습 완료: LinearRegression
모델 저장: kc_house/20260820_083648/linear.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

### 2. Ridge

In [13]:
# 파이프라인 구축 -> 학습 -> 저장
ridge = my_ml.fit_pipeline(
                model=Ridge(random_state=RANDOM_STATE),
                x_train=x_train, y_train=y_train,
                vif=True, 
                scale=True, 
                drop_first=True,
                save_path=f"{workdir}/ridge.pkl")

ridge

대상: 17280행 x 9열 | 모델: Ridge
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0) → 정규화(standard)
  명목형: 더미변수 인코딩(drop_first=True)

모델 학습 완료: Ridge
모델 저장: kc_house/20260820_083648/ridge.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

### 3. Lasso

In [14]:
lasso = my_ml.fit_pipeline(
                model=Lasso(random_state=RANDOM_STATE),
                x_train=x_train, y_train=y_train,
                vif=True, 
                scale=True, 
                drop_first=True,
                save_path=f"{workdir}/lasso.pkl")
lasso

대상: 17280행 x 9열 | 모델: Lasso
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0) → 정규화(standard)
  명목형: 더미변수 인코딩(drop_first=True)

모델 학습 완료: Lasso
모델 저장: kc_house/20260820_083648/lasso.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

### 4. ElasticNet

In [15]:
elasticnet = my_ml.fit_pipeline(
                model=ElasticNet(random_state=RANDOM_STATE),
                x_train=x_train, y_train=y_train,
                vif=True, 
                scale=True, 
                drop_first=True,
                save_path=f"{workdir}/elasticnet.pkl")
elasticnet

대상: 17280행 x 9열 | 모델: ElasticNet
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0) → 정규화(standard)
  명목형: 더미변수 인코딩(drop_first=True)

모델 학습 완료: ElasticNet
모델 저장: kc_house/20260820_083648/elasticnet.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

## #03. 비선형 계열 모델

### 1. KNN

In [16]:
knn = my_ml.fit_pipeline(
                model=KNeighborsRegressor(n_jobs=-1),
                x_train=x_train, y_train=y_train,
                vif=True, 
                scale=True, 
                drop_first=True,
                save_path=f"{workdir}/knn.pkl")

knn

대상: 17280행 x 9열 | 모델: KNeighborsRegressor
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0) → 정규화(standard)
  명목형: 더미변수 인코딩(drop_first=True)

모델 학습 완료: KNeighborsRegressor
모델 저장: kc_house/20260820_083648/knn.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

### 2. Support Vector Regressor

In [17]:
svr = my_ml.fit_pipeline(
                model=SVR(),
                x_train=x_train, y_train=y_train,
                vif=True,
                scale=True, 
                drop_first=True,
                save_path=f"{workdir}/svr.pkl")
svr

대상: 17280행 x 9열 | 모델: SVR
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0) → 정규화(standard)
  명목형: 더미변수 인코딩(drop_first=True)

모델 학습 완료: SVR
모델 저장: kc_house/20260820_083648/svr.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

## #04. 트리 계열 모델

### 1. Decision Tree

In [18]:
dtree = my_ml.fit_pipeline(
                model=DecisionTreeRegressor(random_state=RANDOM_STATE),
                x_train=x_train, y_train=y_train,
                vif=True, 
                encode=True, 
                save_path=f"{workdir}/dtree.pkl")
dtree

대상: 17280행 x 9열 | 모델: DecisionTreeRegressor
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0)
  명목형: 더미변수 인코딩(drop_first=False)

모델 학습 완료: DecisionTreeRegressor
모델 저장: kc_house/20260820_083648/dtree.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

## #05. 앙상블 모델

### 1. Random Forest

In [19]:
rf = my_ml.fit_pipeline(
                model=RandomForestRegressor(
                    random_state=RANDOM_STATE, n_jobs=-1),
                x_train=x_train, y_train=y_train,
                vif=True, 
                encode=True, 
                save_path=f"{workdir}/rf.pkl")
rf

대상: 17280행 x 9열 | 모델: RandomForestRegressor
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0)
  명목형: 더미변수 인코딩(drop_first=False)

모델 학습 완료: RandomForestRegressor
모델 저장: kc_house/20260820_083648/rf.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

### 2. XGBoost

In [20]:
xgb = my_ml.fit_pipeline(
            model=XGBRegressor(
                random_state=RANDOM_STATE, n_jobs=-1),
            x_train=x_train, y_train=y_train,
            vif=True, 
            encode=True, 
            save_path=f"{workdir}/xgb.pkl")
xgb

대상: 17280행 x 9열 | 모델: XGBRegressor
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0)
  명목형: 더미변수 인코딩(drop_first=False)

모델 학습 완료: XGBRegressor
모델 저장: kc_house/20260820_083648/xgb.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

### 3.  LightGBM

In [21]:
lgbm = my_ml.fit_pipeline(
            model=LGBMRegressor(
                    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
            x_train=x_train, y_train=y_train,
            vif=True, 
            encode=True, 
            save_path=f"{workdir}/lgbm.pkl")
lgbm

대상: 17280행 x 9열 | 모델: LGBMRegressor
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0)
  명목형: 더미변수 인코딩(drop_first=False)

모델 학습 완료: LGBMRegressor
모델 저장: kc_house/20260820_083648/lgbm.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

### 4. CatBoost

In [22]:
catboost = my_ml.fit_pipeline(
            model=CatBoostRegressor(
                    random_state=RANDOM_STATE, verbose=0),
            x_train=x_train, y_train=y_train,
            vif=True, 
            encode=False,   # CatBoost는 자체적으로 범주형 처리하므로 encode=False
            save_path=f"{workdir}/catboost.pkl",
            # 이름이 `단계명__인자명` 형식인 인자는 모델의 fit 으로 그대로 전달된다
            model__cat_features=my_qtcheck.get_categorical_column_names(x_train))
catboost

대상: 17280행 x 9열 | 모델: CatBoostRegressor
명목형: ['is_renovated', 'has_basement', 'spatial_cluster']
연속형: ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor']

전처리 단계
  연속형: 다중공선성 제거(VIF >= 10.0)
  명목형: (변환 없음)

모델 학습 완료: CatBoostRegressor
모델 저장: kc_house/20260820_083648/catboost.pkl


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

## #06. 최고 성능 모델 선정

### 1. 학습 모델 파일 목록

In [23]:
# 작업폴더 내의 모든 pkl 파일 목록 확인
model_pickles = gl.glob(f"{workdir}/*.pkl")
print(model_pickles)

['kc_house/20260820_083648/catboost.pkl', 'kc_house/20260820_083648/elasticnet.pkl', 'kc_house/20260820_083648/rf.pkl', 'kc_house/20260820_083648/knn.pkl', 'kc_house/20260820_083648/lasso.pkl', 'kc_house/20260820_083648/ridge.pkl', 'kc_house/20260820_083648/dtree.pkl', 'kc_house/20260820_083648/xgb.pkl', 'kc_house/20260820_083648/linear.pkl', 'kc_house/20260820_083648/svr.pkl', 'kc_house/20260820_083648/lgbm.pkl']


### 2. 학습 모델 불러오기

In [24]:
models = {}                         # 모델명과 모델 객체를 저장할 딕셔너리

for p in model_pickles:
    model_name = p.split(".")[0]    # 파일 목록에서 파일명만 분리
    model = my_ml.load_model(p)     # 모델 로드
    models[model.name_] = model     # 모델명과 모델 객체를 딕셔너리에 저장

# 로드된 모델과 모델 객체의 타입 출력
for name, model in models.items():
    print(f"- {name}: {type(model)}")

- CatBoostRegressor: <class 'sklearn.pipeline.Pipeline'>
- ElasticNet: <class 'sklearn.pipeline.Pipeline'>
- RandomForestRegressor: <class 'sklearn.pipeline.Pipeline'>
- KNeighborsRegressor: <class 'sklearn.pipeline.Pipeline'>
- Lasso: <class 'sklearn.pipeline.Pipeline'>
- Ridge: <class 'sklearn.pipeline.Pipeline'>
- DecisionTreeRegressor: <class 'sklearn.pipeline.Pipeline'>
- XGBRegressor: <class 'sklearn.pipeline.Pipeline'>
- LinearRegression: <class 'sklearn.pipeline.Pipeline'>
- SVR: <class 'sklearn.pipeline.Pipeline'>
- LGBMRegressor: <class 'sklearn.pipeline.Pipeline'>


### 3. 특정 모델의 성능지표 확인

In [25]:
my_ml.reg_score(models['CatBoostRegressor'], x_test, y_test)

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
CatBoostRegressor,0.745,0.199,0.070,0.264,0.019,1.529,-0.067


### 4. 모델간 성능평가 비교

In [26]:
my_ml.reg_compare_models(models,               # 모델 객체들을 담은 딕셔너리
                         x_test,               # 검증 데이터의 독립 변수
                         y_test,               # 검증 데이터의 종속 변수
                         primary="RMSE",       # 주 지표
                         aux=["MAE", "R2"])    # 보조 지표


◆ Score Table Ranking : primary='RMSE', aux=['MAE', 'R2']

▲ step1: 주 지표(RMSE) 기준 정렬 — 낮을수록 좋음 (ASC)
    1. CatBoostRegressor RMSE  =            0.264
    2. LGBMRegressor  RMSE  =            0.265
    3. SVR            RMSE  =            0.269
    4. RandomForestRegressor RMSE  =            0.273
    5. XGBRegressor   RMSE  =            0.273
    6. KNeighborsRegressor RMSE  =            0.286
    7. Ridge          RMSE  =            0.288
    8. LinearRegression RMSE  =            0.288
    9. DecisionTreeRegressor RMSE  =            0.366
   10. ElasticNet     RMSE  =            0.524
   11. Lasso          RMSE  =            0.524

▲ step2: 근소 격차 그룹 묶기 (1등의 5% 이내)
   - 1등 RMSE   : 0.264
   - 허용 범위    : RMSE ≤ 0.278
   - 근소 격차 그룹 (5) : ['CatBoostRegressor', 'LGBMRegressor', 'SVR', 'RandomForestRegressor', 'XGBRegressor']
   - 그룹 외부     (6) : ['KNeighborsRegressor', 'Ridge', 'LinearRegression', 'DecisionTreeRegressor', 'ElasticNet', 'Lasso']

▲ step3: 보조 지표 결정적 결함 점검 (근소 격차 그룹 내부)
  

,Rank,Group,Model,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE,RMSE_Gap
name,,,,,,,,,,,
CatBoostRegressor,1,Contender,CatBoostRegressor,0.745,0.199,0.070,0.264,0.019,1.529,-0.067,0.000
LGBMRegressor,2,Contender,LGBMRegressor,0.744,0.200,0.070,0.265,0.019,1.532,-0.075,0.002
SVR,3,Contender,SVR,0.737,0.201,0.072,0.269,0.019,1.543,-0.035,0.015
RandomForestRegressor,4,Contender,RandomForestRegressor,0.728,0.203,0.074,0.273,0.019,1.557,-0.065,0.032
XGBRegressor,5,Contender,XGBRegressor,0.727,0.206,0.075,0.273,0.019,1.581,-0.074,0.033
KNeighborsRegressor,6,Outside,KNeighborsRegressor,0.701,0.213,0.082,0.286,0.020,1.636,-0.024,0.082
Ridge,7,Outside,Ridge,0.696,0.221,0.083,0.288,0.021,1.697,-0.102,0.091
LinearRegression,8,Outside,LinearRegression,0.696,0.221,0.083,0.288,0.021,1.697,-0.102,0.091
DecisionTreeRegressor,9,Outside,DecisionTreeRegressor,0.512,0.275,0.134,0.366,0.026,2.111,-0.067,0.383


## #08. 하이퍼파라미터 튜닝

### 1. LinearRegressor

In [27]:
# 튜닝 결과가 저장될 위치
output_dir = f"{workdir}_tuned"

In [28]:
%%time

# LinearRegression 은 규제 항이 없어 성능을 조절할 하이퍼파라미터가 사실상 없다.
# 아래 두 옵션이 가질 수 있는 값의 전부라, 이 그리드가 곧 전체 탐색 범위다.
# 즉 "튜닝으로 좋아지지 않는 모델"을 확인하는 것이 이 셀의 목적이다.
param_grid = {
    "model__fit_intercept": [True, False],   # 절편 사용 여부
    "model__positive": [True, False]         # 계수를 양수로 제한할지 여부
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/linear.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/linear.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
LinearRegression,0.696,0.221,0.083,0.288,0.021,1.697,-0.102


CPU times: user 44.1 ms, sys: 13.1 ms, total: 57.2 ms
Wall time: 2.36 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__fit_intercept': [True, False], 'model__positive': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;-

### 2. Ridge

In [29]:
%%time

# [실제 탐색용] 20개 조합
# param_grid = {
#     "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
#     "model__solver": ["auto", "svd", "cholesky", "lsqr"]
# }

# [수업용] 8개 조합. alpha 가 클수록 계수를 0 쪽으로 강하게 눌러 분산을 줄인다.
param_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 100.0],   # 규제 강도
    "model__solver": ["auto", "lsqr"]          # 최적화 알고리즘
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/ridge.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/ridge.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
Ridge,0.696,0.221,0.083,0.288,0.021,1.697,-0.102


CPU times: user 72.1 ms, sys: 22.6 ms, total: 94.7 ms
Wall time: 1.16 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.1, 1.0, ...], 'model__solver': ['auto', 'lsqr']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 :

### 3. Lasso

In [30]:
%%time

# [실제 탐색용] 15개 조합
# param_grid = {
#     "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
#     "model__max_iter": [1000, 5000, 10000]
# }

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/lasso.pkl")

# [수업용] 8개 조합. 기본값 alpha=1.0 은 이 데이터의 타깃 편차(약 0.57)에 비해 너무 강해
# 모든 계수가 0 이 되어 버린다. 작은 alpha 를 넣어야 모델이 살아난다.
param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1],   # 규제 강도 (L1)
    "model__max_iter": [1000, 5000]               # 좌표하강 반복 횟수
}

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/lasso.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
Lasso,0.696,0.221,0.083,0.288,0.021,1.697,-0.102


CPU times: user 73.2 ms, sys: 24.5 ms, total: 97.7 ms
Wall time: 232 ms


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.0001, 0.001, ...], 'model__max_iter': [1000, 5000]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >

### 4. ElasticNet

In [31]:
%%time

# [실제 탐색용] 20개 조합
# param_grid = {
#     "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
#     "model__l1_ratio": [0.1, 0.3, 0.5, 0.9]
# }

# [수업용] 8개 조합. l1_ratio 는 L1(Lasso)과 L2(Ridge)의 혼합 비율로,
# 1 에 가까울수록 Lasso, 0 에 가까울수록 Ridge 처럼 동작한다.
param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1],   # 규제 강도
    "model__l1_ratio": [0.2, 0.8]                 # L1 규제의 비중
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/elasticnet.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/elasticnet.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
ElasticNet,0.696,0.221,0.083,0.288,0.021,1.697,-0.102


CPU times: user 71.4 ms, sys: 24.4 ms, total: 95.8 ms
Wall time: 232 ms


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.0001, 0.001, ...], 'model__l1_ratio': [0.2, 0.8]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 

### 5. KNN

In [32]:
%%time

# [실제 탐색용] 24개 조합
# param_grid = {
#     "model__n_neighbors": [3, 5, 10, 20, 30, 50],
#     "model__weights": ["uniform", "distance"],
#     "model__p": [1, 2]
# }

# [수업용] 8개 조합. 이웃 수가 적으면 과대적합, 많으면 과소적합 쪽으로 기운다.
param_grid = {
    "model__n_neighbors": [5, 10, 20, 30],       # 참조할 이웃 수
    "model__weights": ["uniform", "distance"]    # 거리에 따른 가중 방식
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/knn.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/knn.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
KNeighborsRegressor,0.727,0.206,0.075,0.273,0.019,1.577,0.005


CPU times: user 1.66 s, sys: 36.7 ms, total: 1.7 s
Wall time: 1.02 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': [5, 10, ...], 'model__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displ

### 6. Support Vector Machine

In [33]:
%%time

# [실제 탐색용] 27개 조합 --> 표본 수의 제곱에 비례해 학습 시간이 늘어나므로 매우 오래 걸린다
# param_grid = {
#     "model__C": [0.1, 1.0, 10.0],
#     "model__gamma": ["scale", 0.01, 0.1],
#     "model__epsilon": [0.05, 0.1, 0.2]
# }

# [수업용] SVR 은 이 노트북에서 가장 느린 모델이므로 튜닝 시간을 줄이기 위해 조합을 3개로 축소한다.
param_grid = {
    "model__C": [1.0],           # 오차 허용에 대한 벌점 (클수록 훈련 데이터에 밀착)
    "model__gamma": [0.01],      # RBF 커널의 영향 반경
    "model__epsilon": [0.1]      # 오차 허용 범위 (클수록 훈련 데이터에 밀착)
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/svr.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/svr.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
SVR,0.721,0.210,0.076,0.277,0.020,1.612,-0.017


CPU times: user 5.35 s, sys: 240 ms, total: 5.59 s
Wall time: 10.9 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVR())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [1.0], 'model__epsilon': [0.1], 'model__gamma': [0.01]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2

### 7. Decision Tree

In [34]:
%%time

# [실제 탐색용] 24개 조합
# param_grid = {
#     "model__max_depth": [4, 6, 10, 14, 20, None],
#     "model__min_samples_leaf": [1, 5, 10, 20]
# }

# [수업용] 8개 조합. 기본값(max_depth=None)은 잎이 순수해질 때까지 쪼개
# 훈련 R2 가 1.0 이 되는 전형적인 과대적합 상태다. 깊이를 제한해 이를 완화한다.
param_grid = {
    "model__max_depth": [6, 10, 14, None],   # 트리의 최대 깊이
    "model__min_samples_leaf": [1, 10]       # 잎 노드가 가져야 할 최소 표본 수
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/dtree.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/dtree.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
DecisionTreeRegressor,0.711,0.211,0.079,0.282,0.020,1.619,-0.066


CPU times: user 111 ms, sys: 31.4 ms, total: 142 ms
Wall time: 2.31 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [6, 10, ...], 'model__min_samples_leaf': [1, 10]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2

### 8. Random Forest

In [35]:
%%time

# [실제 탐색용] 18개 조합
# param_grid = {
#     "model__n_estimators": [100, 300, 500],
#     "model__max_depth": [10, 20, None],
#     "model__min_samples_leaf": [1, 5]
# }

# [수업용] 트리를 n_estimators 개 만큼 학습하므로 조합 하나가 비싸다.
# 4개 조합으로 줄였다 (SVR 과 함께 이 노트북에서 오래 걸리는 축에 속한다).
param_grid = {
    "model__n_estimators": [100, 300],     # 숲을 이루는 트리 개수
    "model__min_samples_leaf": [1, 5]      # 잎 노드가 가져야 할 최소 표본 수
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/rf.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/rf.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
RandomForestRegressor,0.739,0.200,0.071,0.267,0.019,1.530,-0.062


CPU times: user 12.4 s, sys: 409 ms, total: 12.8 s
Wall time: 8.77 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...state=3217))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__min_samples_leaf': [1, 5], 'model__n_estimators': [100, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2

### 9. XGBoost

In [36]:
%%time

# [실제 탐색용] 27개 조합
# param_grid = {
#     "model__n_estimators": [300, 600, 1000],
#     "model__max_depth": [3, 4, 6],
#     "model__learning_rate": [0.03, 0.05, 0.1]
# }

# [수업용] 8개 조합. learning_rate 를 낮추면 n_estimators 를 늘려야 하므로
# 두 값은 짝지어 움직인다는 점을 확인한다.
param_grid = {
    "model__n_estimators": [300, 600],        # 부스팅 라운드 수
    "model__max_depth": [4, 6],               # 트리 깊이
    "model__learning_rate": [0.05, 0.1]       # 각 트리의 반영 비율
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/xgb.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/xgb.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
XGBRegressor,0.741,0.201,0.071,0.266,0.019,1.544,-0.071


CPU times: user 834 ms, sys: 7.61 s, total: 8.44 s
Wall time: 3.86 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.05, 0.1], 'model__max_depth': [4, 6], 'model__n_estimators': [300, 600]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and param

### 10.  LightGBM

In [37]:
%%time

# [실제 탐색용] 27개 조합
# param_grid = {
#     "model__n_estimators": [300, 600, 1000],
#     "model__num_leaves": [15, 31, 63],
#     "model__learning_rate": [0.03, 0.05, 0.1]
# }

# [수업용] 8개 조합. LightGBM 은 깊이 대신 잎 개수(num_leaves)로 복잡도를 조절한다.
param_grid = {
    "model__n_estimators": [300, 600],        # 부스팅 라운드 수
    "model__num_leaves": [31, 63],            # 트리 하나가 가질 잎 개수
    "model__learning_rate": [0.05, 0.1]       # 각 트리의 반영 비율
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/lgbm.pkl")

gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다. 이 호출이 있어야 best_estimator_ 가 만들어지고,
# 이후 예측·성능비교가 가능해진다.
gs.fit(x_train, y_train)

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/lgbm.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

gs

,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
LGBMRegressor,0.742,0.200,0.071,0.266,0.019,1.534,-0.076


CPU times: user 3.52 s, sys: 40 s, total: 43.5 s
Wall time: 7min 18s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...verbose=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.05, 0.1], 'model__n_estimators': [300, 600], 'model__num_leaves': [31, 63]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and para

### 11. CatBoost

In [38]:
%%time

# [실제 탐색용] 36개 조합 x 5-fold = CatBoost 학습 180회 --> 수십 분 이상 소요
# param_grid = {
#     "model__iterations": [500, 1000, 2000],
#     "model__depth": [4, 6, 8, 10],
#     "model__learning_rate": [0.03, 0.05, 0.1]
# }

# [수업용] 각 하이퍼파라미터를 일부만 두어 조합을 축소한다.
# 수업 시간에 탐색 과정을 직접 돌려보기 위한 것이므로, 실제 분석에서는 위 범위를 쓴다.
param_grid = {
    "model__iterations": [500, 1000],
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.03, 0.05, 0.1]
}

# 베이스모델 로드
model = my_ml.load_model(f"{workdir}/catboost.pkl")

# 하이퍼파라미터 탐색을 위한 GridSearchCV 객체를 생성
gs = GridSearchCV(estimator=model, param_grid=param_grid,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

# 탐색을 실제로 수행한다.
# CatBoost 는 범주형 컬럼 목록을 fit 시점에 받으므로 여기서도 함께 넘겨야 한다.
gs.fit(x_train, y_train,
       model__cat_features=my_qtcheck.get_categorical_column_names(x_train))

# 튜닝이 완료된 객체에게 모델이름을 부여한다.
gs.name_ = f"{model.name_}_tuned"

# 튜닝이 완료된 모델을 저장한다.
my_ml.save_model(gs, f"{output_dir}/catboost.pkl")

# 성능 평가
display(my_ml.reg_score(gs, x_test, y_test))

/Users/leekh/.pyenv/versions/3.13.9/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE
Model,,,,,,,
CatBoostRegressor,0.747,0.198,0.069,0.263,0.019,1.521,-0.068


CPU times: user 21.4 s, sys: 23.5 s, total: 44.9 s
Wall time: 52.2 s


## #09. 최고 성능 모델 선정

### 1. 학습 모델 파일 목록

In [39]:
# 작업폴더 내의 모든 pkl 파일 목록 확인
model_pickles = gl.glob(f"{output_dir}/*.pkl")
print(model_pickles)

['kc_house/20260820_083648_tuned/catboost.pkl', 'kc_house/20260820_083648_tuned/elasticnet.pkl', 'kc_house/20260820_083648_tuned/rf.pkl', 'kc_house/20260820_083648_tuned/knn.pkl', 'kc_house/20260820_083648_tuned/lasso.pkl', 'kc_house/20260820_083648_tuned/ridge.pkl', 'kc_house/20260820_083648_tuned/dtree.pkl', 'kc_house/20260820_083648_tuned/xgb.pkl', 'kc_house/20260820_083648_tuned/linear.pkl', 'kc_house/20260820_083648_tuned/svr.pkl', 'kc_house/20260820_083648_tuned/lgbm.pkl']


### 2. 학습 모델 불러오기

In [40]:
models = {}

for p in model_pickles:
    # 파일 목록에서 파일명만 분리
    model_name = p.split(".")[0]

    # 모델 로드
    model = my_ml.load_model(p)

    # 모델명과 모델 객체를 딕셔너리에 저장
    models[model.name_] = model

for name, model in models.items():
    print(f"- {name}: {type(model)}")

- CatBoostRegressor_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- ElasticNet_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- RandomForestRegressor_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- KNeighborsRegressor_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- Lasso_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- Ridge_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- DecisionTreeRegressor_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- XGBRegressor_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- LinearRegression_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- SVR_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>
- LGBMRegressor_tuned: <class 'sklearn.model_selection._search.GridSearchCV'>


### 3. 모델간 성능평가 비교

In [41]:
my_ml.reg_compare_models(models,               # 모델 객체들을 담은 딕셔너리
                         x_test,               # 검증 데이터의 독립 변수
                         y_test,               # 검증 데이터의 종속 변수
                         primary="RMSE",       # 주 지표
                         aux=["MAE", "R2"])    # 보조 지표


◆ Score Table Ranking : primary='RMSE', aux=['MAE', 'R2']

▲ step1: 주 지표(RMSE) 기준 정렬 — 낮을수록 좋음 (ASC)
    1. CatBoostRegressor_tuned RMSE  =            0.263
    2. LGBMRegressor_tuned RMSE  =            0.266
    3. XGBRegressor_tuned RMSE  =            0.266
    4. RandomForestRegressor_tuned RMSE  =            0.267
    5. KNeighborsRegressor_tuned RMSE  =            0.273
    6. SVR_tuned      RMSE  =            0.277
    7. DecisionTreeRegressor_tuned RMSE  =            0.282
    8. Lasso_tuned    RMSE  =            0.288
    9. ElasticNet_tuned RMSE  =            0.288
   10. LinearRegression_tuned RMSE  =            0.288
   11. Ridge_tuned    RMSE  =            0.288

▲ step2: 근소 격차 그룹 묶기 (1등의 5% 이내)
   - 1등 RMSE   : 0.263
   - 허용 범위    : RMSE ≤ 0.276
   - 근소 격차 그룹 (5) : ['CatBoostRegressor_tuned', 'LGBMRegressor_tuned', 'XGBRegressor_tuned', 'RandomForestRegressor_tuned', 'KNeighborsRegressor_tuned']
   - 그룹 외부     (6) : ['SVR_tuned', 'DecisionTreeRegressor_tuned', 'Lasso_tune

,Rank,Group,Model,R2,MAE,MSE,RMSE,RMSLE,MAPE,MPE,RMSE_Gap
name,,,,,,,,,,,
CatBoostRegressor_tuned,1,Contender,CatBoostRegressor,0.747,0.198,0.069,0.263,0.019,1.521,-0.068,0.000
LGBMRegressor_tuned,2,Contender,LGBMRegressor,0.742,0.200,0.071,0.266,0.019,1.534,-0.076,0.009
XGBRegressor_tuned,3,Contender,XGBRegressor,0.741,0.201,0.071,0.266,0.019,1.544,-0.071,0.012
RandomForestRegressor_tuned,4,Contender,RandomForestRegressor,0.739,0.200,0.071,0.267,0.019,1.530,-0.062,0.015
KNeighborsRegressor_tuned,5,Contender,KNeighborsRegressor,0.727,0.206,0.075,0.273,0.019,1.577,0.005,0.038
SVR_tuned,6,Outside,SVR,0.721,0.210,0.076,0.277,0.020,1.612,-0.017,0.050
DecisionTreeRegressor_tuned,7,Outside,DecisionTreeRegressor,0.711,0.211,0.079,0.282,0.020,1.619,-0.066,0.070
Lasso_tuned,8,Outside,Lasso,0.696,0.221,0.083,0.288,0.021,1.697,-0.102,0.096
ElasticNet_tuned,9,Outside,ElasticNet,0.696,0.221,0.083,0.288,0.021,1.697,-0.102,0.096


### 문제 풀이

#### 1. 팀장이 요구한 비율대로 데이터를 나눴을 때, 모델이 학습에 사용한 관측치는 몇 건인가요? (정수, 단위: 건)

- **정답**: `17280`
- **복습 개념**: 훈련·검증 데이터 분리입니다. 모델의 성능은 **학습에 쓰지 않은 데이터**에서 재야 합니다. 학습에 쓴 데이터로 점수를 매기면 답안지를 보고 채점하는 셈이라, 외운 만큼 점수가 올라갑니다. 머신러닝에서는 독립변수를 feature, 종속변수를 target이라 부르고, 분리한 결과는 훈련용 독립·검증용 독립·훈련용 종속·검증용 종속 네 덩어리가 됩니다.
- **풀이 접근**: 먼저 데이터에서 종속변수인 가격 열만 떼어 target으로 두고, 나머지를 feature로 둡니다. 그다음 검증 비율을 0.2로 지정해 네 덩어리로 나누고, 각 덩어리의 행·열 수를 찍어 확인합니다. 난수 씨앗을 고정해야 다음에 다시 돌려도 같은 분할이 나옵니다.
- **근거(계산 결과)**: 전체는 21,600건 × 10열이고 가격을 떼어 내면 feature는 **9열**입니다. 여기에 검증 비율 0.2를 적용해 훈련 **(17280, 9)**, 검증 **(4320, 9)** 로 갈렸습니다. 21,600 × 0.8 = 17,280 이 그대로 맞아떨어집니다.
- **자주 하는 실수**: 열 수를 10으로 답하기 쉽습니다. 가격은 맞혀야 할 정답이므로 독립변수 쪽에 남아 있으면 안 됩니다. 만약 남아 있었다면 모든 모델의 결정계수가 1.0에 가깝게 나왔을 것이고, 그건 성능이 좋은 게 아니라 **정답을 입력으로 넣은 것**입니다. 하나 더, 이번은 가격을 맞히는 회귀 문제라 분할 시 계층 지정을 하지 않았습니다. 집단 비율을 맞춰 나누는 계층 분할은 분류 문제에서 씁니다.

#### 2. 기본값으로 학습한 11개 모델을 주 지표 기준으로 줄 세운 뒤, 1등과 사실상 차이가 없다고 한 묶음으로 분류된 후보 그룹에는 모델이 몇 개 들어갔나요? (정수, 단위: 개)

- **정답**: `5`
- **복습 개념**: 모델 간 성능 비교와 순위 판정 절차입니다. 지표 하나의 소수점 셋째 자리로 우열을 가르지 않고, ① 주 지표로 정렬 → ② **1등과의 격차가 근소한 모델을 한 그룹으로 묶고** → ③ 그 그룹 안에서 보조 지표에 결정적 결함이 있는지 확인 → ④ 최종 순위를 확정하는 네 단계를 밟습니다. RMSE는 오차 지표라 **낮을수록** 좋고, 결정계수는 설명력 지표라 높을수록 좋습니다.
- **풀이 접근**: 저장해 둔 모델 파일들을 한 번에 불러와 모델명을 열쇠로 하는 딕셔너리에 담습니다. 그 딕셔너리와 검증 데이터를 넘겨 주 지표를 RMSE로, 보조 지표를 MAE와 결정계수로 지정한 비교표를 만듭니다. 출력의 2단계에 묶인 그룹 구성원 수를 셉니다.
- **근거(계산 결과)**: 1등 CatBoost의 RMSE가 **0.264**이고, 이 값의 5% 이내인 **0.278 이하**가 허용 범위로 잡혔습니다. 여기에 들어온 모델은 CatBoost 0.264 · LightGBM 0.265 · SVR 0.269 · RandomForest 0.273 · XGBoost 0.273 로 **5개**입니다. KNN 0.286부터는 그룹 밖입니다. 그룹 안 5개는 보조 지표 결함 점검(MAE 0.219 초과 또는 결정계수 0.695 미만)에서도 모두 결함 0개였습니다.
- **헷갈리기 쉬운 점**: "1등이 나왔으니 그걸 쓰면 된다"고 끝내기 쉽습니다. 1등과 2등의 RMSE 차이는 **0.002**입니다. 데이터를 다시 나누거나 난수 씨앗만 바꿔도 뒤집힐 수 있는 크기라, 이 정도 격차를 실력 차이로 읽으면 안 됩니다. 그래서 후보를 묶어 두고 **속도 · 재현성 · 해석 가능성** 같은 다른 기준을 함께 놓고 고르는 것입니다. 반대로 하위권은 격차의 성격이 다릅니다. 결정트리 0.366, 라쏘와 엘라스틱넷 0.524는 근소한 차이가 아니라 **다른 이야기**입니다.

#### 3. 규제 계열 두 모델은 기본값으로 학습했을 때 결정계수가 0.000, 즉 평균값만 답한 것과 다름없는 상태였는데 튜닝 후 다른 선형 모델과 같은 수준으로 돌아왔습니다. 이 회복을 만들어 낸 하이퍼파라미터의 이름은 무엇인가요? (파라미터 이름 1개)

- **정답**: `alpha`
- **복습 개념**: 규제(regularization)의 강도 조절입니다. 라쏘와 엘라스틱넷은 계수를 0 쪽으로 눌러 과대적합을 막는데, 그 누르는 힘의 세기가 `alpha`입니다. 이 값이 너무 크면 모든 계수가 0이 되어 모델은 **어떤 입력을 받아도 평균만 답하는 상수 예측기**가 됩니다. 그때 결정계수는 0이 됩니다.
- **풀이 접근**: 비교표에서 결정계수가 0.000인 모델을 먼저 찾아 두 모델의 공통점을 봅니다. 둘 다 L1 규제를 쓴다는 점을 확인했으면, 규제 강도의 기본값과 종속변수의 흩어진 정도를 견줍니다. 그다음 탐색 격자에 기본값보다 훨씬 작은 강도를 넣어 다시 학습시키고 성능이 살아나는지 봅니다.
- **근거(계산 결과)**: 기본값 `alpha=1.0` 에서 라쏘와 엘라스틱넷은 나란히 RMSE **0.524**, 결정계수 **−0.000**, MAE 0.414 로 11개 중 공동 꼴찌였습니다. 이 0.524 라는 값은 로그 가격의 표준편차(약 0.57)와 거의 같습니다. 즉 **아무 설명도 하지 않고 평균만 찍은 것과 같은 오차**입니다. 탐색 격자에 0.0001 ~ 0.1 의 작은 값을 넣어 다시 돌리자 두 모델 모두 RMSE **0.288**, 결정계수 **0.696** 이 되어 선형회귀·릿지와 완전히 같은 성적으로 돌아왔습니다.
- **함께 생각해 볼 점**: 여기서 릿지는 왜 멀쩡했을까요. 릿지의 L2 규제는 계수를 0에 **가깝게** 줄일 뿐 정확히 0으로 만들지는 못합니다. 반면 라쏘의 L1 규제는 계수를 **딱 0으로** 잘라 내므로, 강도가 세면 변수를 전부 탈락시켜 모델이 텅 비게 됩니다. 엘라스틱넷은 두 규제를 섞은 것이라 라쏘 쪽 성질을 물려받아 같이 무너졌습니다. **성능이 0으로 나왔다고 "이 모델은 이 데이터에 안 맞는다"고 결론 내리면 안 됩니다.** 맞고 안 맞고를 판단하기 전에 기본값이 이 데이터의 눈금에 맞는지부터 확인해야 합니다. 규제 강도는 종속변수의 크기와 흩어진 정도에 따라 적정값이 달라지는데, 이번 종속변수는 로그를 씌워 0.57 수준으로 작아져 있었습니다.

#### 4. 튜닝을 마치고 다시 비교했더니, 기본값일 때는 상위 후보 그룹 안에 있었는데 튜닝 후 오히려 성능이 떨어져 그룹 밖으로 밀려난 모델이 하나 있습니다. 어느 모델인가요? (모델명 1개)

- **정답**: `SVR`
- **복습 개념**: 격자 탐색의 한계입니다. 격자 탐색은 **당신이 적어 준 후보 조합 안에서만** 최선을 고릅니다. 격자 안에 기본값이 들어 있지 않으면, 기본값보다 못한 조합이라도 그중 1등이 뽑혀 저장됩니다. 게다가 교차검증 점수는 **훈련 데이터를 5겹으로 나눠** 잰 값이므로, 검증 데이터에서의 최종 성적과 항상 같은 방향으로 움직이지도 않습니다.
- **풀이 접근**: 튜닝 전 비교표와 튜닝 후 비교표를 나란히 놓고 모델별 RMSE와 순위를 대응시킵니다. 값이 커진(나빠진) 모델을 찾고, 그중 후보 그룹 소속이 바뀐 모델을 골라냅니다. 그다음 그 모델에 어떤 후보 조합을 넣었는지 되짚어 원인을 확인합니다.
- **근거(계산 결과)**: **SVR**은 튜닝 전 RMSE **0.269**로 전체 3위이자 후보 그룹 안에 있었는데, 튜닝 후 **0.277**로 나빠져 **6위 · 그룹 밖**이 되었습니다. 결정계수도 0.737 → 0.721로 내려갔습니다. 원인은 격자에 있습니다. 학습 시간을 줄이려고 후보를 `C=1.0`, `gamma=0.01`, `epsilon=0.1` **단 한 조합**으로 잡았는데, 여기서 `gamma` 기본값은 `scale`이라 0.01과 다릅니다. 즉 탐색이랄 것도 없이 **기본값보다 나쁜 조합 하나로 갈아 끼운 것**입니다. LightGBM도 0.265 → 0.266으로 아주 조금 나빠졌지만 그룹 안 순위는 지켰습니다.
- **자주 하는 실수**: "튜닝했으니 당연히 좋아졌겠지"라고 믿고 튜닝 후 표만 보고하는 것입니다. 튜닝은 **탐색 범위를 잘 잡았을 때만** 좋아집니다. 그래서 격자에는 기본값을 함께 넣어 두는 편이 안전하고, 튜닝 전후를 반드시 같은 검증 데이터로 비교해 봐야 합니다. 이번 노트북에서 SVR과 랜덤포레스트의 격자를 크게 줄인 것은 수업 시간 안에 돌리기 위한 타협이며, 실제 분석에서는 주석으로 남겨 둔 넓은 범위를 써야 합니다.

#### 5. 튜닝까지 마친 11개 모델을 최종 비교했을 때 1위로 보고하게 되는 모델은 무엇인가요? (모델명 1개)

- **정답**: `CatBoostRegressor`
- **복습 개념**: 최종 모델 선정입니다. 튜닝된 모델들을 다시 한자리에 모아 **같은 검증 데이터**로 재평가하고, 주 지표 정렬 → 근소 격차 그룹 → 보조 지표 결함 점검의 같은 절차를 한 번 더 밟습니다. 모델을 파일로 저장해 두는 이유가 여기 있습니다. 학습을 다시 하지 않고도 언제든 불러와 같은 조건에서 비교할 수 있기 때문입니다.
- **풀이 접근**: 튜닝 결과가 저장된 폴더의 파일 목록을 훑어 모델을 모두 불러오고, 튜닝 전과 똑같은 지표 설정으로 비교표를 만듭니다. 최종 순위표의 1행을 읽되, 근소 격차 그룹에 누가 함께 들어 있는지도 같이 확인합니다.
- **근거(계산 결과)**: **CatBoost**가 RMSE **0.263** · MAE **0.198** · 결정계수 **0.747** 로 1위이고, 보조 지표 결함도 없습니다. 뒤이어 LightGBM 0.266 · XGBoost 0.266 · RandomForest 0.267 · KNN 0.273 까지 5개가 근소 격차 그룹(허용 범위 0.276 이하)에 묶였습니다. CatBoost는 튜닝 전에도 1위(0.264)였으니 튜닝으로 얻은 것은 **0.001**뿐입니다.
- **실무 포인트**: 정작 튜닝의 효과가 컸던 쪽은 하위권이었습니다. 결정트리는 기본값이 잎이 순수해질 때까지 쪼개는 전형적인 과대적합 상태였는데 깊이를 제한하자 결정계수가 0.512 → 0.711로 회복되었고, 라쏘·엘라스틱넷도 되살아났습니다(개선폭은 6번에서 직접 계산해 봅니다). **이미 잘 맞는 모델은 튜닝으로 더 얻을 것이 적고, 망가져 있던 모델일수록 튜닝의 몫이 큽니다.** 튜닝은 순위를 뒤집는 도구라기보다 **모델이 제 실력을 내게 만드는** 도구에 가깝습니다.
- **결론**: 팀장에게 드릴 답은 이렇게 정리됩니다. **최종 채택은 `CatBoostRegressor`이고, 검증 RMSE는 0.263입니다.** 다만 2~4위와의 격차가 0.004 이내이므로 "압도적 1위"라고 쓰면 안 되고, **부스팅 계열 세 모델이 사실상 동률이며 그중 학습 안정성과 범주형 처리 편의를 고려해 CatBoost를 골랐다**고 적는 것이 정확합니다. 한 가지 더 덧붙일 것이 있습니다. 이 0.263은 **로그를 씌운 가격**의 오차라 달러가 아닙니다. 보고서에는 반드시 원 단위로 되돌린 해석을 함께 실어야 하는데, 그 환산은 7번에서 직접 해 봅니다.

#### 6. 튜닝 전에 결정계수가 0이었던 두 모델을 빼고 보면, 튜닝으로 RMSE가 가장 많이 낮아진 모델이 하나 있습니다. 그 모델의 RMSE는 튜닝 전보다 얼마나 낮아졌나요? (소수 셋째 자리)

- **정답**: `0.084`
- **복습 개념**: 튜닝 효과의 크기를 재는 방법입니다. "튜닝했다"는 사실이 아니라 **같은 검증 데이터에서 주 지표가 얼마나 움직였는가**로 효과를 말해야 합니다. 그리고 그 효과는 모델마다 크게 다릅니다. 기본값이 이미 잘 맞는 모델은 얻을 것이 적고, 기본값이 **과대적합이나 과소적합 상태로 망가져 있던** 모델일수록 튜닝의 몫이 큽니다.
- **풀이 접근**: 튜닝 전 비교표와 튜닝 후 비교표를 모델명 기준으로 붙여 RMSE 두 열을 나란히 만들고, 뺄셈으로 개선폭 열을 하나 더 만듭니다. 결정계수가 0이었던 두 모델은 조건으로 걸러 낸 뒤 개선폭이 가장 큰 행을 찾습니다. 그다음 그 모델에 어떤 후보 조합을 넣었는지 되짚어 왜 그렇게 좋아졌는지 확인합니다.
- **근거(계산 결과)**: 제외 대상은 라쏘와 엘라스틱넷(둘 다 0.524 → 0.288, 개선폭 0.236)입니다. 남은 아홉 개 중 1위는 **결정트리**로 **0.366 → 0.282**, 개선폭 **0.084**입니다. 결정계수도 0.512 → 0.711로 올랐습니다. 2위인 KNN이 0.286 → 0.273으로 0.013이니 **여섯 배 넘는 격차**입니다. 나머지는 XGBoost 0.007, 랜덤포레스트 0.006, CatBoost 0.001이고 선형회귀·릿지는 0.000, LightGBM과 SVR은 오히려 음수입니다.
- **함께 생각해 볼 점**: 결정트리의 기본 설정은 깊이 제한이 없어 **잎이 순수해질 때까지** 계속 쪼갭니다. 훈련 데이터는 거의 완벽히 맞히지만 검증 데이터에서는 무너지는 전형적인 과대적합이고, 튜닝 전 결정계수 0.512가 그 결과입니다. 깊이와 잎 최소 표본 수를 제한하자 0.711까지 올라왔습니다. 그런데도 최종 순위는 7위에 머뭅니다. **과대적합을 고쳤다고 해서 앙상블을 이기지는 못합니다.** 트리 하나가 할 수 있는 일에는 한계가 있고, 그 한계를 넘으려고 트리를 여러 개 묶은 것이 랜덤포레스트와 부스팅 계열입니다. 결정트리는 성능으로 채택할 모델이라기보다, **앙상블이 왜 필요한지를 보여 주는 기준선**으로 읽는 편이 맞습니다.

#### 7. 최종 채택 모델의 검증 RMSE는 로그를 씌운 가격에 대한 오차입니다. 이 오차를 원래 가격 단위로 되돌리면 실제 가격의 몇 배에 해당하나요? (소수 둘째 자리, 단위: 배)

- **정답**: `1.30`
- **복습 개념**: 로그 변환한 종속변수의 성능 해석입니다. 지표는 **변환된 눈금 위에서** 계산되므로 RMSE 0.263은 달러가 아닙니다. 로그 공간에서의 **차이**는 원 단위에서의 **비율**에 해당하므로, 되돌릴 때는 빼기가 아니라 지수를 취해 배수로 읽어야 합니다.
- **풀이 접근**: 먼저 이 데이터의 로그가 자연로그인지 확인합니다. 종속변수의 최솟값과 최댓값에 지수를 취해 원래 가격대와 맞는지 보면 알 수 있습니다. 자연로그임이 확인되면 최종 모델의 RMSE에 지수를 취해 배수를 구하고, 거기서 1을 빼 몇 퍼센트 오차인지로 바꿔 읽습니다.
- **근거(계산 결과)**: 로그 가격의 최솟값 11.225에 지수를 취하면 **75,000달러**, 최댓값 15.857은 **7,700,000달러**로 실제 가격대와 정확히 맞아 자연로그임이 확인됩니다. 최종 모델의 RMSE 0.263에 지수를 취하면 e^0.263 ≈ **1.30**입니다. 즉 전형적인 예측 오차가 **실제 가격의 약 1.30배 안팎**이라는 뜻입니다.
- **실무 포인트**: 이 1.30을 방향까지 풀면 위로는 약 **+30%**, 아래로는 약 **−23%**(1 ÷ 1.30)입니다. 로그 공간에서는 대칭이지만 원 단위로 돌아오면 **위아래가 비대칭**이 됩니다. 그래서 "평균 ±30만 달러 오차"처럼 고정 금액으로 쓰면 틀립니다. 5억짜리 집과 50억짜리 집의 오차 금액이 같을 수 없기 때문입니다. **로그를 씌워 학습한 모델의 오차는 금액이 아니라 비율로 보고합니다.**
- **결론**: 이제 팀장에게 드릴 최종 보고 문장이 완성됩니다. **"11개 모델을 같은 조건에서 비교해 CatBoost를 채택했고, 검증 RMSE는 로그 기준 0.263 — 원 단위로는 실제 가격의 약 1.3배, 즉 ±30% 수준의 오차입니다."** 여기에 두 가지를 덧붙입니다. 2~4위와의 격차가 0.004 이내라 **부스팅 계열은 사실상 동률**이라는 점, 그리고 튜닝은 상위권 순위를 뒤집지 못했고 대신 **망가져 있던 하위 모델을 제자리로 돌려놓았다**는 점입니다.